# M3: direct CLV influence in LightGCN propagation

Dunnhumby seed-42 validation screen. The experiment compares M1, N-only, V-only, CLV, and shuffled-CLV arms. It preserves the binary edge set and M1 user-from-item coefficients, and redistributes only each item's fixed incoming message mass among its customer neighbors. Test and holdout are disabled.

## Setup

Mount Drive and check out the reviewed source commit. Required inputs are the Dunnhumby raw CSV files under `/content/drive/MyDrive/논문/data/raw/dunnhumby/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = 'ea9ae87c674162d3dab403a7357abf3a36ad5978'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

## Preflight checks

Confirm GPU availability, the fixed seed-42 validation scope, `MIN_ITEM_INTER=1`, and disabled test/holdout evaluation before training.

In [ ]:
import json, torch
from lightgcn_clv_m3_mass_preserving import (
    configure_m3_clv_influence_dunnhumby_run,
    preflight_summary,
    run_experiment,
)

cfg = configure_m3_clv_influence_dunnhumby_run()
assert torch.cuda.is_available(), 'Select a GPU runtime before running.'
summary = preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['split'] == 'validation only'
assert summary['shared_invariants']['min_item_inter'] == 1
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

## Run

Train M1 and the four fixed M3 arms. Native diagnostics and comparison files are saved to Drive.

In [ ]:
result_df = run_experiment(cfg)

## Results and decision

Report general accuracy, purchase-value-weighted hit decomposition, recommended price percentile, and exposure safeguards together. This one-seed validation screen does not establish population significance.

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'revenue@10', 'revenue@20', 'revenue@50',
    'mean_hits@10', 'hit_value@10', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment',
]
available = [column for column in columns if column in result_df.columns]
validation = result_df[result_df['split'].eq('val')][available]
display(validation.sort_values('model_id'))
print('Screening decision:')
print(json.dumps(result_df.attrs['screening_decision'], ensure_ascii=False, indent=2))
print('Result files:', result_df.attrs['result_paths'])